In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import sys

sys.path.insert(0, "../..")

In [ ]:
import chex
import dill
import jax
import jax.numpy as jnp
import json
import numpy as np
import pandas as pd

from flax import nnx
from torch.utils.data import DataLoader
from typing import Any
from typing_extensions import Protocol, runtime_checkable

from src.dataset import get_iter
from src.datasets.sum import Addition
from src.utils import parse_dict

In [ ]:
Dtype = Any
Shape = tuple[int, ...]

@runtime_checkable
class HasCache(Protocol):
    def init_cache(self, input_shape: Shape, dtype: Dtype = jnp.float32): ...

def evaluate(learner_path, num_batches):
    config_dict = json.load(open(os.path.join(learner_path, "config.json"), "r"))

    # Load model
    last_step = sorted(os.listdir(os.path.join(learner_path, "models")))[-1]
    train_state = dill.load(
        open(os.path.join(learner_path, "models", last_step), "rb")
    )
    model = nnx.merge(
        train_state.graphdef,
        train_state.params,
        train_state.rest,
    )

    half_precision = config_dict["half_precision"]
    eval_seed = 42
    batch_size = 1

    dtype = jnp.bfloat16 if half_precision else jnp.float32

    rng = jax.random.PRNGKey(eval_seed)
    rng, _ = jax.random.split(rng)

    # Get dataset
    dataset_kwargs = parse_dict(config_dict["dataset_kwargs"])
    dataset = Addition(
        dataset_kwargs.context_len,
        dataset_kwargs.max_int * 2,
        False,
        config_dict["seeds"]["data_seed"],
        "question_only_decode",
        dataset_kwargs.train_val_ratio,
        getattr(dataset_kwargs, "right_to_left", False),
    )
    data_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=0,
    )
    data_iter = get_iter(data_loader, None, dtype)

    # Sample next batch and evaluate
    max_decode_len = dataset_kwargs.context_len

    model.eval()
    for _path, m in model.iter_modules():
        if isinstance(m, HasCache):
            input_shape = (
                1,
                max_decode_len,
                config_dict["model_config"]["model_kwargs"]["embed_dim"],
            )
            m.init_cache(input_shape, dtype=dtype)

    graphdef, _, rest = nnx.split(model, nnx.Cache, ...)
    def decode(batch, cache):
        module = nnx.merge(graphdef, cache, rest)
        module.set_attributes(deterministic=True, decode=True)
        out = module(batch)
        cache = nnx.state(module, nnx.Cache)
        return out, cache

    all_res = []
    for _ in range(num_batches):
        batch = next(data_iter)
        cache = nnx.state(model, nnx.Cache)
        preds = []
        for step_i in range(max_decode_len):
            if step_i < batch["sequence"].shape[1]:
                res, cache = decode({
                    "sequence": batch["sequence"][:, [step_i]]
                }, cache)
            else:
                res, cache = decode({
                    "sequence": preds[-1],
                }, cache)
            preds.append(jnp.argmax(res, axis=-1))
        all_res.append((batch, preds))
    return all_res

In [ ]:
base_path = "/home/bryanpu1/projects/iclr_2026/icl_architecture/scaling_jax/results"

algo_name = "addition"
# run_name = "max_int_8-08-13-25_14_25_20-d8f77477-f3d7-4972-b749-4358930b8829"
# run_name = "cot-max_int_32-08-18-25_14_13_45-0caf324f-9daf-456c-a309-edabd8ee526d"
run_name = "max_int_32-08-18-25_14_09_01-22351d80-4697-4c8e-bc1b-2095cd04f288"
# run_name = "right_to_left-max_int_32-08-18-25_14_09_30-e7ed1fa3-ca51-493e-b823-3f30cdcacc4a"
# run_name = "gru-cot-max_int_32-08-19-25_11_14_35-c2cd8587-d274-449c-a72f-0020f183466e"
# run_name = "gru-max_int_32-08-19-25_11_15_05-7d3c2ae9-5f86-4c13-a303-193d55dab2ad"
# run_name = "gru-right_to_left-max_int_32-08-19-25_11_15_27-0f476f45-6c4c-45c7-939a-00064327f3ab"

learner_path = os.path.join(base_path, algo_name, run_name)
all_res = evaluate(
    learner_path,
    num_batches=16,
)

In [ ]:
for sample_i, (batch, preds) in enumerate(all_res):
    print("Sample: {} ----------------".format(sample_i))
    print(".    Input: {}".format(batch["sequence"].flatten().tolist()))
    print(".   Target: {}".format(batch["target"].flatten().tolist()))

    pred_trace = jnp.concatenate(preds, axis=-1).flatten()[
        batch["sequence"].shape[1] - 1:# batch["sequence"].shape[1] + batch["target"].shape[1]
    ]

    # Print the prediction trace
    curr_trace = []
    for output in pred_trace:
        if output == 3 and len(curr_trace) > 0:
            print(".    Trace: {}".format(curr_trace))
            curr_trace = []
        else:
            curr_trace.append(output.item())
        
        if output == 4:
            print("Prediction: {}".format(curr_trace))
            break